In [72]:
import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import OneHotEncoder
from sklearn.linear_model import LogisticRegression


In [73]:
# Custom Transformer for Feature Engineering

class TitanicFeatureTransformer(BaseEstimator, TransformerMixin):
    def fit(self, X, y=None):
        return self
    
    def transform(self, X):
        X = X.copy()

        # Travelling with Family/Solo
        X['With_Family'] = ((X['Parch'] + X['SibSp']) > 0).astype(int)

        # Extract Title
        X['Title'] = X['Name'].str.extract('([A-Za-z]+)\.', expand=False)
        X['Title'] = X['Title'].replace(['Ms','Lady'],'Miss')
        titles_to_keep = ['Master', 'Miss', 'Mrs', 'Mr']
        X['Title'] = X['Title'].apply(lambda x: x if x in titles_to_keep else 'Other')

        # Cabin Category
        X['Cabin_Category'] = X['Cabin'].apply(lambda x: 'Unknown' if pd.isna(x) else x[0])
        cabins_to_keep = ['C','B','D','E','A','F', 'Unknown']
        X['Cabin_Category'] = X['Cabin_Category'].apply(lambda x: x if x in cabins_to_keep else 'Other')
        X['Cabin_Missing'] = X['Cabin_Category'].apply(lambda x: 1 if x == 'Unknown' else 0)

        # Interaction Features
        X['Pclass_Sex'] = X['Pclass'].astype(str) + '_' + X['Sex']


        # Drop unused columns
        X = X.drop(['Name', 'Ticket', 'Cabin', 'PassengerId'], axis=1)

        return X

In [74]:
pipeline_num_features = Pipeline(
    steps=[
        ('impute', SimpleImputer(strategy='median', add_indicator=True)),
        ('scale', StandardScaler())
    ]
)

pipeline_cat_features = Pipeline(
    steps=[
        ('impute', SimpleImputer(strategy='most_frequent')),
        ('ohe', OneHotEncoder(drop='first'))
    ]
)

preprocessor = ColumnTransformer(
    [
        ('num_feature', pipeline_num_features, ['Age', 'Fare']),
        ('cat_feature', pipeline_cat_features, ['Embarked', 'Sex', 'Pclass', 'Title', 'Cabin_Category', 'Pclass_Sex'])
    ],
    remainder='passthrough'
)

In [75]:
from sklearn.ensemble import RandomForestClassifier
# Building Final Pipeline
full_pipeline = Pipeline(
    steps = [
        ('feature_eng', TitanicFeatureTransformer()),
        ('preprocessing', preprocessor),
        #('model', LogisticRegression())
        ('model', RandomForestClassifier())
    ]
)

In [76]:
X = pd.read_csv('./data/train.csv')

y = X['Survived']
X = X.drop(['Survived'], axis=1)

In [77]:
# Train-test split
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [45]:
X_train.head()

,PassengerId,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
331,332,1,"Partner, Mr. Austen",male,45.5,0,0,113043,28.5000,C124,S
733,734,2,"Berriman, Mr. William John",male,23.0,0,0,28425,13.0000,NaN,S
382,383,3,"Tikkanen, Mr. Juho",male,32.0,0,0,STON/O 2. 3101293,7.9250,NaN,S
704,705,3,"Hansen, Mr. Henrik Juul",male,26.0,1,0,350025,7.8542,NaN,S
813,814,3,"Andersson, Miss. Ebba Iris Alfrida",female,6.0,4,2,347082,31.2750,NaN,S


In [26]:
## Transform
#feature_transformer = TitanicFeatureTransformer()
#X_train_trf = feature_transformer.fit_transform(X_train)
#X_test_trf = feature_transformer.transform(X_test)

In [79]:
# Train and predict
#full_pipeline.fit(X_train, y_train)
#predictions = full_pipeline.predict(X_test)

In [80]:
#from sklearn.metrics import accuracy_score
#print(accuracy_score(y_test, predictions))

In [78]:
from sklearn.model_selection import cross_val_score

# Use your current pipeline (with SibSp, Parch kept)
scores = cross_val_score(full_pipeline, X, y, cv=5, scoring='accuracy')

print(f"CV Scores: {scores}")
print(f"Mean Accuracy: {scores.mean():.4f}")
print(f"Std Dev: {scores.std():.4f}")

CV Scores: [0.81564246 0.80337079 0.8258427  0.74157303 0.83707865]
Mean Accuracy: 0.8047
Std Dev: 0.0335
